In [0]:
# Load the CSV file into a Spark DataFrame
file_path = "/mnt/fsdh-dbk-main-mount/Sample_contractHistoryComplete-contratsOctroyesComplet.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

# Display the DataFrame
display(df)

In [0]:
from pyspark.sql.functions import col, upper, regexp_replace, when, coalesce, lit, count, sum as spark_sum, mean, min as spark_min, max as spark_max, first, monotonically_increasing_id
from datetime import datetime

class VendorDatasetCreator:
    def __init__(self, df):
        self.raw_data = df
        self.vendor_data = None
        self.final_dataset = None

    def extract_vendor_fields(self):
        vendor_fields = {
            'legal_name': 'supplierLegalName-nomLegalFournisseur-eng',
            'standardized_name': 'supplierStandardizedName-nomNormaliseFournisseur-eng',
            'operating_name': 'supplierOperatingName-nomCommercialFournisseur-eng',
            'employee_count': 'supplierEmployeeCount-fournisseurNombreEmployes-eng',
            'address_line': 'supplierAddressLine-ligneAdresseFournisseur-eng',
            'city': 'supplierAddressCity-fournisseurAdresseVille-eng',
            'province': 'supplierAddressProvince-fournisseurAdresseProvince-eng',
            'postal_code': 'supplierAddressPostalCode-fournisseurAdresseCodePostal',
            'country': 'supplierAddressCountry-fournisseurAdressePays-eng',
            'contact_email': 'contactInfoEmail-informationsContactCourriel',
            'contact_name': 'contactInfoName-informationsContactNom',
            'contract_amount': 'contractAmount-montantContrat',
            'contract_currency': 'contractCurrency-contratMonnaie',
            'contract_award_date': 'contractAwardDate-dateAttributionContrat',
            'contract_number': 'contractNumber-numeroContrat',
            'procurement_category': 'procurementCategory-categorieApprovisionnement'
        }
        available_fields = {key: field for key, field in vendor_fields.items() if field in self.raw_data.columns}
        self.vendor_data = self.raw_data.select([col(v).alias(k) for k, v in available_fields.items()])
        return self.vendor_data

    def clean_vendor_names(self):
        def clean_name(col_name):
            return upper(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
                regexp_replace(col(col_name), r'\bINCORPORATED\b', 'INC'), r'\bLIMITED\b', 'LTD'),
                r'\bCORPORATION\b', 'CORP'), r'\bCOMPANY\b', 'CO'), r'\bENTERPRISES\b', 'ENT'),
                r'[.,]+$', '')).alias(col_name + '_clean')

        self.vendor_data = self.vendor_data.withColumn('legal_name_clean', clean_name('legal_name')) \
                                           .withColumn('standardized_name_clean', clean_name('standardized_name')) \
                                           .withColumn('operating_name_clean', clean_name('operating_name')) \
                                           .withColumn('master_name', coalesce(
                                               col('standardized_name_clean'),
                                               col('legal_name_clean'),
                                               col('operating_name_clean')))
        return self.vendor_data

    def standardize_addresses(self):
        province_mapping = {
            'BRITISH COLUMBIA': 'BC', 'BC': 'BC',
            'ALBERTA': 'AB', 'AB': 'AB',
            'SASKATCHEWAN': 'SK', 'SK': 'SK',
            'MANITOBA': 'MB', 'MB': 'MB',
            'ONTARIO': 'ON', 'ON': 'ON',
            'QUEBEC': 'QC', 'QC': 'QC', 'QUÉBEC': 'QC',
            'NEW BRUNSWICK': 'NB', 'NB': 'NB',
            'NOVA SCOTIA': 'NS', 'NS': 'NS',
            'PRINCE EDWARD ISLAND': 'PE', 'PE': 'PE', 'PEI': 'PE',
            'NEWFOUNDLAND AND LABRADOR': 'NL', 'NL': 'NL',
            'NORTHWEST TERRITORIES': 'NT', 'NT': 'NT',
            'NUNAVUT': 'NU', 'NU': 'NU',
            'YUKON': 'YT', 'YT': 'YT'
        }
        # Use a UDF for province mapping
        from pyspark.sql.functions import udf
        from pyspark.sql.types import StringType
        def map_province(prov):
            if prov is None:
                return None
            return province_mapping.get(prov.upper(), prov.upper())
        map_province_udf = udf(map_province, StringType())

        self.vendor_data = self.vendor_data.withColumn('postal_code_clean', 
                                                       when(col('postal_code').isNotNull(), 
                                                            upper(regexp_replace(col('postal_code'), r'\s|-', '')))) \
                                           .withColumn('province_clean', map_province_udf(upper(col('province'))))
        return self.vendor_data

    def categorize_vendor_size(self):
        def categorize_size(employee_count_col):
            return when(employee_count_col < 20, 'Small (1-19)') \
                   .when(employee_count_col < 100, 'Medium (20-99)') \
                   .when(employee_count_col < 500, 'Large (100-499)') \
                   .otherwise('Very Large (500+)')
        self.vendor_data = self.vendor_data.withColumn('vendor_size_category', categorize_size(col('employee_count')))
        return self.vendor_data

    def aggregate_contract_metrics(self):
        self.vendor_data = self.vendor_data.withColumn('contract_amount_numeric', col('contract_amount').cast('double')) \
                                           .withColumn('contract_award_date', col('contract_award_date').cast('date'))

        agg_metrics = self.vendor_data.groupBy('master_name').agg(
            count('contract_amount_numeric').alias('total_contracts'),
            spark_sum('contract_amount_numeric').alias('total_contract_value'),
            mean('contract_amount_numeric').alias('avg_contract_value'),
            spark_min('contract_award_date').alias('first_contract_date'),
            spark_max('contract_award_date').alias('most_recent_contract_date'),
            first('procurement_category').alias('primary_category')
        )
        return agg_metrics

    def deduplicate_vendors(self):
        # Use isNull checks for completeness_score
        self.vendor_data = self.vendor_data.withColumn(
            'completeness_score',
            (col('legal_name').isNotNull().cast('int') +
             col('address_line').isNotNull().cast('int') +
             col('city').isNotNull().cast('int') +
             col('province').isNotNull().cast('int') +
             col('contact_email').isNotNull().cast('int'))
        )
        unique_vendors = self.vendor_data.orderBy(col('completeness_score').desc()).dropDuplicates(['master_name'])
        return unique_vendors

    def create_final_dataset(self):
        self.extract_vendor_fields()
        self.clean_vendor_names()
        self.standardize_addresses()
        self.categorize_vendor_size()
        unique_vendors = self.deduplicate_vendors()
        contract_metrics = self.aggregate_contract_metrics()
        self.final_dataset = unique_vendors.join(contract_metrics, 'master_name', 'left')
        self.final_dataset = self.final_dataset.withColumn('created_date', lit(datetime.now().strftime('%Y-%m-%d'))) \
                                               .withColumn('vendor_id', monotonically_increasing_id())
        return self.final_dataset

# Example usage
creator = VendorDatasetCreator(df)
vendor_dataset = creator.create_final_dataset()
display(vendor_dataset)

In [0]:
# Load the CSV file into a Spark DataFrame
file_path = "/mnt/fsdh-dbk-main-mount/contracts_sample 1.csv"
df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

# Display the DataFrame
display(df_new)